In [ ]:
#!pip install transformers torch scikit-learn requests -q

In [ ]:
import torch
import json
import requests
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

RAHUL_PROFILE = """
Name: Rahul Rubugunday
Degree: M.S. Computer Science, Northern Arizona University (May 2026) + MCA from SASTRA University
Experience: 2+ years Software Engineer at Tristha Global (Chennai)
  - Built backend microservices using Python, Java, Node.js, Express, TypeScript
  - Worked on a low-code/no-code automation platform built on Node-RED
  - Enterprise clients: Amex, Visa, Fiserv
  - Bulk field-update feature (350+ test cases), PDF report export (500+ scenarios)
  - AWS deployment, Azure Blob Storage
Projects:
  - Fake News Detection: BiLSTM + attention mechanism, 92% accuracy, 40K articles (PyTorch, NLP)
  - Movie Recommendation: SentenceTransformers + FAISS, 45K movies, sub-10ms retrieval
Skills: Python, Java, Node.js, Express, REST APIs, Microservices, Docker, AWS, Azure,
        MongoDB, Redis, Git, SQL, PyTorch, FAISS, SentenceTransformers, System Design, OOP
MLE resume strengths: ML projects, PyTorch, NLP, embeddings, model architecture
SWE resume strengths: backend microservices, production reliability, REST APIs, AWS, Tristha work
"""

MLE_KEYWORDS = [
    "machine learning", "deep learning", "ml", "ai", "nlp", "computer vision",
    "neural", "model", "research", "data science", "pytorch", "tensorflow",
    "embedding", "transformer", "llm", "scientist"
]

print("Imports done.")

Imports done.


In [ ]:
print("Loading ModernBERT... (this takes ~1 min first time)")

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
model = AutoModel.from_pretrained("answerdotai/ModernBERT-base")
model.eval()

print(f"ModernBERT loaded. Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

Loading ModernBERT... (this takes ~1 min first time)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ModernBERT loaded. Device: GPU


In [ ]:
def get_embedding(text: str):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to("cuda" if torch.cuda.is_available() else "cpu")

    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    return embedding


def recommend_resume(role: str) -> str:
    role_lower = role.lower()
    if any(kw in role_lower for kw in MLE_KEYWORDS):
        return "MLE"
    return "SWE"


def build_reasoning(score: int, resume: str, company: str, role: str) -> str:
    if score >= 75:
        fit = "strong fit"
    elif score >= 50:
        fit = "moderate fit"
    else:
        fit = "weak fit"
    return f"{company} — {role} is a {fit} at {score}% similarity. Recommend {resume} resume."


print("Functions ready.")

Functions ready.


In [ ]:
LISTINGS_URL = "https://raw.githubusercontent.com/SimplifyJobs/New-Grad-Positions/dev/.github/scripts/listings.json"

print("Fetching jobs from Simplify...")
response = requests.get(LISTINGS_URL, timeout=10)
response.raise_for_status()
jobs_raw = response.json()

active_jobs = [
    {
        "id": job.get("id", ""),
        "company": job.get("company_name", "Unknown"),
        "role": job.get("title", "Unknown"),
        "location": ", ".join(job.get("locations", ["Remote"])),
        "url": job.get("url", ""),
        "active": job.get("active", False),
        "date_updated": job.get("date_updated", 0),
    }
    for job in jobs_raw
    if job.get("active", False)
][:10]

print(f"Fetched {len(active_jobs)} active jobs. Scoring with ModernBERT...")

profile_embedding = get_embedding(RAHUL_PROFILE)

results = []
for job in active_jobs:
    job_text = f"{job['company']} {job['role']} {job['location']}"
    job_embedding = get_embedding(job_text)
    similarity = cosine_similarity(profile_embedding, job_embedding)[0][0]
    score = max(0, min(100, round(float(similarity) * 100)))
    resume = recommend_resume(job["role"])
    reasoning = build_reasoning(score, resume, job["company"], job["role"])
    results.append({**job, "score": score, "resume": resume, "reasoning": reasoning})
    print(f"  {job['company']} — {job['role']} — {score}% — {resume}")

print(f"\nDone. {len(results)} jobs scored.")

Fetching jobs from Simplify...
Fetched 10 active jobs. Scoring with ModernBERT...
  NorthMark Strategies — Graduate Engineering Program — 91% — SWE
  Adobe — University Graduate - Machine Learning Engineer — 91% — MLE
  Nextdoor — Machine Learning Engineer – New Grad 2026 - Machine Learning - Nextdoor — 92% — MLE
  JPMorganChase — Quantitative Research - Data Analytics - Associate — 91% — MLE
  Figma — Software Engineer — 92% — SWE
  mthree — Junior Software Developer - Go Lang — 91% — SWE
  Notion — Software Engineer, Fullstack — 92% — SWE
  Western Alliance Bancorporation — Engineer I - AI Business Engineer — 88% — MLE
  Wolverine Trading — C++ Software Engineer — 92% — SWE
  Medtronic — Software Operations Engineer I — 90% — SWE

Done. 10 jobs scored.


In [ ]:
from google.colab import files
import json

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved results.json")
files.download("results.json")

Saved results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>